# PG-Stack · Step 5b — Low-label deep dive + new hospital with few labels

Step 5 found the graph prior's added value sits at the **low-label end** (5 %: M3 − M1 = +0.0095, Holm p < 0.001). This notebook makes that claim robust:

**A2 — dense low-label curve:** 1 %, 2 %, 5 %, 10 % of training patients, **10 repeats** each. Tests (Holm across fractions): M3 vs M1 (primary), **M2 vs M1**, M3 vs M2, M2 vs M0.

**B2 — new hospital with a few labels** (most realistic deployment): for each hospital with ≥ 10 cancers, train on the 6 other hospitals **+ 0 / 5 / 10 / 25 % of the target hospital's patients**, test on a **fixed 75 %** of that hospital. 3 repeats. Summarised with a **random-effects meta-analysis** across hospitals (DerSimonian–Laird, I²).

Same models and features as Step 5 (M0 lesion · M1 + global · M2 + graph prior · M3 + both · U zero-label).
CPU is enough. Runtime ≈ 60–80 min. Resumable.

In [ ]:
import os, sys, glob, time, gc, warnings
import numpy as np, pandas as pd, lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
COMP_DIR = "/kaggle/input/competitions/isic-2024-challenge"
SEED = 42
FRACTIONS = [0.01, 0.02, 0.05, 0.10]
REPEATS_A = 10
TARGET_SHARES = [0.0, 0.05, 0.10, 0.25]
REPEATS_B = 3
N_BAGS = 3
NEG_PER_POS = 20
N_BOOT = 100
MIN_POS_HOSPITAL = 10

def locate(name):
    if os.path.exists(f"{WORK}/{name}"): return f"{WORK}/{name}"
    for d in range(1, 5):
        hits = glob.glob("/kaggle/input/" + "*/" * d + name)
        if hits: return hits[0]
    return None
paths = {n: locate(n) for n in ["oof_step3.csv", "emb_pca32.npy", "pgstack_eval.py"]}
assert all(paths.values()), f"missing: {[k for k, v in paths.items() if v is None]}"
sys.path.insert(0, os.path.dirname(paths["pgstack_eval.py"]))
from pgstack_eval import *
print(paths)

## 1–3. Features: lesion, global, graph prior (identical to Step 5)

In [ ]:
oof = pd.read_csv(paths["oof_step3.csv"])
df = pd.read_csv(f"{COMP_DIR}/train-metadata.csv", low_memory=False)
df = oof[["isic_id", "fold"]].merge(df, on="isic_id", how="left")
assert (df.isic_id.values == oof.isic_id.values).all()
emb = np.load(paths["emb_pca32.npy"]).astype(np.float32)
assert len(emb) == len(oof)

LEAK = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
        "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
NUM_RAW = [c for c in df.columns if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]

def add_lesion_features(d):
    d = d.copy(); eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"] = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"] = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"] = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"] = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"] = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"] = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"] = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"] = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"] = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"] = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"] = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"] = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"] = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"] = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast", "lesion_color_difference",
       "border_complexity", "color_uniformity", "position_distance_3d", "perimeter_to_area_ratio",
       "area_to_perimeter_ratio", "lesion_visibility_score", "symmetry_border_consistency",
       "color_shape_composite", "log_area", "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast"]
df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS: df[c] = df[c].fillna("NA").astype("category")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = int(folds.max()) + 1
POS_COLS = ["tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "position_distance_3d"]
TAB_COLS = [c for c in LESION_NUM if c not in POS_COLS]        # appearance/shape only, for the tabular graph
print(len(df), "lesions |", len(LESION_NUM), "lesion features")

In [ ]:
df["hospital"] = df["attribution"].fillna("unknown").astype(str)
print(df.groupby("hospital").agg(lesions=("isic_id", "size"), cancers=("target", "sum"),
                                 patients=("patient_id", "nunique")).sort_values("cancers", ascending=False))

In [ ]:
g = df.groupby("patient_id"); GLOBAL = []
for c in LESION_NUM:
    df[f"{c}_pnorm"] = (df[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6); GLOBAL.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size"); GLOBAL.append("patient_n_lesions")

In [ ]:
X = df[LESION_NUM].astype(np.float32); X = X.fillna(X.median()).values
tab_idx = [LESION_NUM.index(c) for c in TAB_COLS]
TAB = StandardScaler().fit_transform(X[:, tab_idx]).clip(-6, 6).astype(np.float32)
TAB10 = PCA(10, random_state=SEED).fit_transform(TAB).astype(np.float32)
XYZ = df[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].fillna(0).values.astype(np.float32)
SP = {"tab": TAB, "app": emb, "pos": XYZ}
n = len(df)
GP = {f"gp_{s}_k{k}_{t}": np.full(n, np.nan, np.float32) for s in SP for k in [5, 10, 20] for t in ["dist", "lof"]}
GP["gp_mahal"] = np.full(n, np.nan, np.float32); GP["gp_iforest"] = np.full(n, np.nan, np.float32)

def lof(dist, idx, k):
    d, i = dist[:, :k], idx[:, :k]; reach = np.maximum(d, d[:, -1][i])
    lrd = 1.0 / (reach.mean(1) + 1e-6); return lrd[i].mean(1) / lrd

t0 = time.time()
for j, (p, rows) in enumerate(pd.Series(np.arange(n)).groupby(pid).indices.items()):
    r = np.asarray(rows); m = len(r)
    if m < 6: continue
    km = min(20, m - 1)
    for s, Z in SP.items():
        dist, idx = NearestNeighbors(n_neighbors=km + 1).fit(Z[r]).kneighbors(Z[r]); dist, idx = dist[:, 1:], idx[:, 1:]
        for k in [5, 10, 20]:
            kk = min(k, km); GP[f"gp_{s}_k{k}_dist"][r] = dist[:, :kk].mean(1); GP[f"gp_{s}_k{k}_lof"][r] = lof(dist, idx, kk)
    if m >= 15:
        Zp = TAB10[r]; dz = Zp - Zp.mean(0); C = np.cov(Zp, rowvar=False) + 1e-3 * np.eye(10)
        GP["gp_mahal"][r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(C), dz))
    if m >= 10:
        Zi = np.c_[TAB10[r], emb[r, :10]]
        GP["gp_iforest"][r] = -IsolationForest(n_estimators=100, random_state=SEED).fit(Zi).score_samples(Zi)
    if j % 200 == 0: print(f"  patient {j} | {time.time()-t0:.0f}s")
for k, v in GP.items(): df[k] = v
key = df.patient_id.astype(str) + "|" + df.anatom_site_general.astype(str)
df["gp_pos_k10_dist_siteadj"] = df["gp_pos_k10_dist"] / (df.groupby(key)["gp_pos_k10_dist"].transform("median") + 1e-6)
PRIOR = list(GP.keys()) + ["gp_pos_k10_dist_siteadj"]
print(f"graph prior: {len(PRIOR)} features in {time.time()-t0:.0f}s")

# zero-label score: average percentile rank of the 3 strongest label-free scores
U = np.nanmean(np.stack([pd.Series(df[c]).rank(pct=True).values for c in ["gp_tab_k10_lof", "gp_mahal", "gp_iforest"]]), 0)
U = np.nan_to_num(U, nan=0.5)
print("zero-label U score: AUROC", round(roc_auc_score(y, U), 4), "| pAUC80", round(pauc_80(y, U), 4))

## 4. Trainer
Same LightGBM recipe; `min_child_samples` scales with the training-set size (`n/50`, clipped to 5–40) so tiny label budgets can still split. The rule is identical for every model, so comparisons stay fair.

In [ ]:
BASE_F = LESION_NUM + CAT_COLS
FSETS = {"M0_lesion": BASE_F, "M1_global": BASE_F + GLOBAL,
         "M2_prior": BASE_F + PRIOR, "M3_global_prior": BASE_F + GLOBAL + PRIOR}
MODELS = list(FSETS)
XALL = {m: df[f] for m, f in FSETS.items()}
BASE_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7,
                   feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                   n_estimators=300, verbosity=-1)

def fit_predict(model, tr_rows, te_rows, seed):
    pos, neg = tr_rows[y[tr_rows] == 1], tr_rows[y[tr_rows] == 0]
    Xte = XALL[model].iloc[te_rows]; pred = np.zeros(len(te_rows))
    for b in range(N_BAGS):
        rng = np.random.default_rng(seed + b)
        sub = np.r_[pos, rng.choice(neg, min(len(neg), NEG_PER_POS * len(pos)), replace=False)]
        mcs = int(np.clip(len(sub) / 50, 5, 40))
        m = lgb.LGBMClassifier(**BASE_PARAMS, min_child_samples=mcs, random_state=seed + b).fit(XALL[model].iloc[sub], y[sub])
        pred += m.predict_proba(Xte)[:, 1] / N_BAGS
    return pred

pat = df.groupby("patient_id").agg(fold=("fold", "first"), has_pos=("target", "max"),
                                   hospital=("hospital", lambda s: s.mode().iat[0]))
rows_of = pd.Series(np.arange(n)).groupby(pid).indices

def sample_patients(ids_df, share, rng):
    chosen = []
    for hp in [0, 1]:
        ids = ids_df.index[ids_df.has_pos == hp].values
        k = int(round(share * len(ids)))
        if hp == 1: k = max(1, k)                      # at least one cancer patient
        if k > 0: chosen += list(rng.choice(ids, min(k, len(ids)), replace=False))
    return chosen

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.zeros(len(p)); run = 0
    for rank, i in enumerate(o): run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj

## 5. A2 — dense low-label learning curve (10 repeats)

In [ ]:
A_PATH = f"{WORK}/step5b_lowlabel.npz"
LA = dict(np.load(A_PATH)) if os.path.exists(A_PATH) else {}
t0 = time.time()
for q in FRACTIONS:
    for rep in range(REPEATS_A):
        if all(f"{m}|{q}|{rep}" in LA for m in MODELS): continue
        preds = {m: np.zeros(n, np.float32) for m in MODELS}; n_pos = []
        for f in range(N_FOLDS):
            rng = np.random.default_rng(SEED + 7919 * rep + 101 * f + int(q * 10000))
            chosen = sample_patients(pat[pat.fold != f], q, rng)
            tr = np.concatenate([rows_of[p] for p in chosen]); te = np.where(folds == f)[0]
            n_pos.append(int(y[tr].sum()))
            for m in MODELS: preds[m][te] = fit_predict(m, tr, te, SEED + 13 * f + 1000 * rep)
        for m in MODELS: LA[f"{m}|{q}|{rep}"] = preds[m]
        LA[f"npos|{q}|{rep}"] = np.array(n_pos)
        np.savez_compressed(A_PATH, **LA)
        print(f"q={q:.2f} rep={rep}: cancers/fold={np.mean(n_pos):.0f}  " +
              "  ".join(f"{m.split('_')[0]}={pauc_80(y, preds[m]):.4f}" for m in MODELS) + f"  ({(time.time()-t0)/60:.1f} min)")

In [ ]:
U_score = pauc_80(y, U)
rows = []
for q in FRACTIONS:
    r = dict(fraction=q, cancers_per_fold=np.mean([LA[f"npos|{q}|{k}"].mean() for k in range(REPEATS_A)]))
    for m in MODELS:
        v = np.array([pauc_80(y, LA[f"{m}|{q}|{k}"]) for k in range(REPEATS_A)])
        r[m] = v.mean(); r[f"{m}_sd"] = v.std()
    rows.append(r)
la = pd.DataFrame(rows)
la["U_zero_label"] = U_score
for a, b in [("M3_global_prior", "M1_global"), ("M2_prior", "M1_global"), ("M3_global_prior", "M2_prior"), ("M2_prior", "M0_lesion")]:
    la[f"{a.split('_')[0]}-{b.split('_')[0]}"] = la[a] - la[b]
print(la[["fraction", "cancers_per_fold"] + MODELS + ["U_zero_label", "M3-M1", "M2-M1", "M3-M2", "M2-M0"]].round(4).to_string(index=False))

In [ ]:
# patient bootstrap, every (model, fraction, repeat) scored once per resample; deltas averaged over repeats
t0 = time.time()
boot = PatientBootstrap(pid, n_boot=N_BOOT, seed=SEED)
picks = [i for i in boot.samples if y[i].sum() > 0]
SA = {(m, q): np.array([[pauc_80(y[i], LA[f"{m}|{q}|{k}"][i]) for k in range(REPEATS_A)] for i in picks])
      for m in MODELS for q in FRACTIONS}                       # shape (resamples, repeats)
print(f"bootstrap {time.time()-t0:.0f}s")

COMPS = [("M3_global_prior", "M1_global", "PRIMARY"), ("M2_prior", "M1_global", "secondary"),
         ("M3_global_prior", "M2_prior", "secondary"), ("M2_prior", "M0_lesion", "secondary")]
out = []
for a, b, fam in COMPS:
    block = []
    for q in FRACTIONS:
        d = (SA[(a, q)] - SA[(b, q)]).mean(1)                  # averaged over the 10 training subsets
        rep_delta = np.array([pauc_80(y, LA[f"{a}|{q}|{k}"]) - pauc_80(y, LA[f"{b}|{q}|{k}"]) for k in range(REPEATS_A)])
        block.append(dict(family=fam, comparison=f"{a} vs {b}", fraction=q, delta=round(d.mean(), 4),
                          ci95=f"[{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]",
                          p=(d <= 0).mean(), repeats_positive=f"{(rep_delta > 0).sum()}/{REPEATS_A}"))
    for r, pa in zip(block, holm([r["p"] for r in block])): r["p_holm"] = round(pa, 4); r["sig_0.05"] = pa < 0.05
    out += block
la_tests = pd.DataFrame(out); la_tests["p"] = la_tests.p.round(4)
print(la_tests.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
style = {"M0_lesion": ("lesion only", "#8c8c8c", "o"), "M1_global": ("+ global context", "#1f77b4", "s"),
         "M2_prior": ("+ graph prior", "#2ca02c", "^"), "M3_global_prior": ("+ global + graph prior", "#d62728", "D")}
prev = locate("results_step5_learning_curve.csv")
prev = pd.read_csv(prev) if prev else None
for m, (lab, colr, mk) in style.items():
    ax.errorbar(la.fraction * 100, la[m], yerr=la[f"{m}_sd"], label=lab, color=colr, marker=mk, capsize=3)
    if prev is not None and m in prev.columns:
        p2 = prev[prev.fraction >= 0.25]
        ax.plot(p2.fraction * 100, p2[m], color=colr, marker=mk, ls=":", mfc="none")
ax.axhline(U_score, color="#9467bd", ls="--", label="zero-label graph score")
ax.set_xscale("log"); ax.set_xticks([1, 2, 5, 10, 25, 50, 100]); ax.set_xticklabels(["1%", "2%", "5%", "10%", "25%", "50%", "100%"])
ax.set_xlabel("share of training patients with labels"); ax.set_ylabel("pAUC above 80% TPR (max 0.2)")
ax.set_title("Label efficiency (solid: Step 5b, 10 repeats; dotted: Step 5)"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/fig_lowlabel_curve.png", dpi=200); plt.show()

## 6. B2 — new hospital with a few labels
For each hospital with ≥ 10 cancers: fix a random 75 % of its patients as the **test set**; the remaining 25 % is a pool from which 0 / 5 / 10 / 25 % of the hospital's patients are added to training (nested). Training always includes all other hospitals.

In [ ]:
HOSP = df.hospital.values
hosp_pos = df.groupby("hospital").target.sum()
HOSPS = sorted(hosp_pos[hosp_pos >= MIN_POS_HOSPITAL].index)
B_PATH = f"{WORK}/step5b_target.npz"
LB = dict(np.load(B_PATH)) if os.path.exists(B_PATH) else {}
t0 = time.time()
for h in HOSPS:
    ph = pat[pat.hospital == h]
    for rep in range(REPEATS_B):
        rng = np.random.default_rng(SEED + 31 * rep + 1009 * HOSPS.index(h))   # deterministic -> resumable
        pool = sample_patients(ph, 0.25, rng)                         # 25 % pool (stratified)
        test_p = [p for p in ph.index if p not in set(pool)]
        rng.shuffle(pool)
        te = np.concatenate([rows_of[p] for p in test_p])
        LB[f"te|{h}|{rep}"] = te
        other = np.where(HOSP != h)[0]
        for t in TARGET_SHARES:
            if all(f"{m}|{h}|{rep}|{t}" in LB for m in MODELS): continue
            k = int(round(t / 0.25 * len(pool)))
            add = pool[:k]
            tr = np.r_[other, np.concatenate([rows_of[p] for p in add])] if k > 0 else other
            for m in MODELS: LB[f"{m}|{h}|{rep}|{t}"] = fit_predict(m, tr, te, SEED + 77 * rep).astype(np.float32)
            np.savez_compressed(B_PATH, **LB)
        print(f"{h[:40]:40s} rep={rep} test cancers={int(y[te].sum()):3d}  ({(time.time()-t0)/60:.1f} min)")

In [ ]:
rows = []
for h in HOSPS:
    for t in TARGET_SHARES:
        r = dict(hospital=h[:35], target_share=t)
        for m in MODELS:
            r[m] = np.mean([pauc_80(y[LB[f"te|{h}|{k}"]], LB[f"{m}|{h}|{k}|{t}"]) for k in range(REPEATS_B)])
        rows.append(r)
lb = pd.DataFrame(rows)
macro = lb.groupby("target_share")[MODELS].mean()
macro["M3-M1"] = macro.M3_global_prior - macro.M1_global; macro["M2-M1"] = macro.M2_prior - macro.M1_global
macro["M2-M0"] = macro.M2_prior - macro.M0_lesion
print("Macro-average pAUC80 over", len(HOSPS), "hospitals:"); print(macro.round(4).to_string())
print(); print(lb.pivot(index="hospital", columns="target_share", values="M3_global_prior").round(4).to_string())

### Random-effects meta-analysis across hospitals (DerSimonian–Laird)

In [ ]:
def dersimonian_laird(est, var):
    est, var = np.asarray(est), np.asarray(var); w = 1 / var
    fixed = (w * est).sum() / w.sum(); Q = (w * (est - fixed) ** 2).sum(); dfq = len(est) - 1
    tau2 = max(0.0, (Q - dfq) / (w.sum() - (w ** 2).sum() / w.sum()))
    ws = 1 / (var + tau2); mu = (ws * est).sum() / ws.sum(); se = np.sqrt(1 / ws.sum())
    I2 = max(0.0, (Q - dfq) / Q) if Q > 0 else 0.0
    from scipy.stats import norm
    return mu, mu - 1.96 * se, mu + 1.96 * se, 1 - norm.cdf(mu / se), I2

t0 = time.time(); meta = []
for a, b in [("M3_global_prior", "M1_global"), ("M2_prior", "M1_global"), ("M2_prior", "M0_lesion")]:
    for t in TARGET_SHARES:
        est, var = [], []
        for h in HOSPS:
            te0 = LB[f"te|{h}|0"]; pids = pid[te0]
            bb = PatientBootstrap(pids, n_boot=N_BOOT, seed=SEED)
            ds = []
            for i in bb.samples:
                yi = y[te0][i]
                if yi.sum() == 0: continue
                ds.append(pauc_80(yi, LB[f"{a}|{h}|0|{t}"][i]) - pauc_80(yi, LB[f"{b}|{h}|0|{t}"][i]))   # test-sampling variance (repeat 0)
            obs = np.mean([pauc_80(y[LB[f"te|{h}|{k}"]], LB[f"{a}|{h}|{k}|{t}"]) - pauc_80(y[LB[f"te|{h}|{k}"]], LB[f"{b}|{h}|{k}|{t}"])
                           for k in range(REPEATS_B)])
            est.append(obs); var.append(np.var(ds) + 1e-8)
        mu, lo, hi, p, I2 = dersimonian_laird(est, var)
        meta.append(dict(comparison=f"{a} vs {b}", target_share=t, pooled_delta=round(mu, 4),
                         ci95=f"[{lo:+.4f}, {hi:+.4f}]", p_one_sided=round(p, 4), I2=round(I2, 2),
                         hospitals_positive=f"{int((np.array(est) > 0).sum())}/{len(est)}"))
meta = pd.DataFrame(meta)
meta["p_holm"] = np.nan
for c in meta.comparison.unique():
    m_ = meta.comparison == c; meta.loc[m_, "p_holm"] = holm(meta.loc[m_, "p_one_sided"].values).round(4)
print(meta.to_string(index=False)); print(f"({time.time()-t0:.0f}s)")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for m, (lab, colr, mk) in style.items():
    ax.plot(macro.index * 100, macro[m], label=lab, color=colr, marker=mk)
ax.set_xlabel("share of the NEW hospital's patients labelled (%)"); ax.set_ylabel("macro pAUC80 over hospitals")
ax.set_title("New hospital with a few labels"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/fig_new_hospital.png", dpi=200); plt.show()

In [ ]:
la.to_csv(f"{WORK}/results_step5b_lowlabel.csv", index=False); la_tests.to_csv(f"{WORK}/tests_step5b_lowlabel.csv", index=False)
lb.to_csv(f"{WORK}/results_step5b_target_per_hospital.csv", index=False); macro.to_csv(f"{WORK}/results_step5b_target_macro.csv")
meta.to_csv(f"{WORK}/tests_step5b_meta.csv", index=False)
print("saved. Download results_step5b_*, tests_step5b_*, fig_lowlabel_curve.png, fig_new_hospital.png")